# AI Use Case 3 — Enum: classificação, roteamento e guardrails em agentes

**Problema:** agentes tomam decisões **categóricas** o tempo todo — qual a intenção do usuário, para qual time/agente rotear,
qual modelo usar, que ferramentas um agente pode chamar, por que o modelo parou. Se essas categorias forem strings livres,
o LLM inventa variações (`"Billing"`, `"billing_issue"`, `"cobrança"`) e o seu `if` não casa.

**Ideia central:** o Enum é a **fonte única do vocabulário**, usada ao mesmo tempo para:

```
                    ┌──────────────► prompt   (descrição de cada rótulo gerada do Enum)
                    │
   class Intent(Enum) ─────────────► schema   (Pydantic → "enum": [...] → decodificação restrita)
                    │
                    ├──────────────► código   (dict[Intent, handler], match/case exaustivo)
                    │
                    └──────────────► métricas (contagem por rótulo, matriz de confusão)
```

Roteiro:
1. Classificação de intenção com rótulos ricos (descrição no próprio Enum)
2. Roteamento com tabela de despacho e limiar de confiança
3. Multi-rótulo com `Flag`
4. Seleção de modelo por tier (Enum com metadados de custo)
5. Guardrails: capacidades do agente com `Flag`
6. Enums "abertos" para respostas do provedor (`stop_reason`)
7. Checklist

In [1]:
import json
from enum import Enum, StrEnum, Flag, auto
from functools import reduce
from operator import or_
from typing import Callable, Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from fake_llm import ScriptedLLM, keyword_llm

## 1. Classificação de intenção com rótulos ricos

Cada membro carrega a **descrição** usada no prompt. Adicionar uma intenção nova = adicionar um membro;
prompt, schema e roteamento se atualizam juntos.

In [2]:
class Intent(StrEnum):
    def __new__(cls, value: str, description: str):
        obj = str.__new__(cls, value)
        obj._value_ = value
        obj.description = description
        return obj

    BILLING = ("billing", "Cobranças, faturas, reembolso, forma de pagamento")
    TECH_SUPPORT = ("tech_support", "Erro no produto, bug, não consegue acessar, lentidão")
    SALES = ("sales", "Interesse em comprar, planos, preços, demonstração")
    CANCELLATION = ("cancellation", "Quer cancelar a assinatura ou encerrar a conta")
    OTHER = ("other", "Qualquer coisa que não se encaixe nas anteriores")

def label_guide(enum_cls: type[Enum]) -> str:
    return "\n".join(f"- {m.value}: {m.description}" for m in enum_cls)

class Classification(BaseModel):
    model_config = ConfigDict(extra="forbid")
    intent: Intent = Field(description="Rótulo da intenção principal")
    confidence: float = Field(ge=0, le=1, description="Confiança de 0 a 1")
    rationale: str = Field(max_length=200, description="Justificativa curta")

SYSTEM = f"""Classifique a mensagem do cliente em UMA intenção:
{label_guide(Intent)}
Responda só com JSON no schema: {json.dumps(Classification.model_json_schema(), ensure_ascii=False)}"""

print(f"SYSTEM[:420]: {SYSTEM[:420]} ...")
print(f'\nenum no schema: {Classification.model_json_schema()["$defs"]["Intent"]["enum"]}')

SYSTEM[:420]: Classifique a mensagem do cliente em UMA intenção:
- billing: Cobranças, faturas, reembolso, forma de pagamento
- tech_support: Erro no produto, bug, não consegue acessar, lentidão
- sales: Interesse em comprar, planos, preços, demonstração
- cancellation: Quer cancelar a assinatura ou encerrar a conta
- other: Qualquer coisa que não se encaixe nas anteriores
Responda só com JSON no schema: {"$defs": {"Intent": {"enu ...

enum no schema: ['billing', 'tech_support', 'sales', 'cancellation', 'other']


Simulamos o classificador com um LLM de palavras-chave (determinístico). Repare no último caso: o "modelo" devolve
`"Cobrança"` — fora do vocabulário. A validação pega; numa aplicação real isso dispararia o retry com feedback do notebook 01
(ou nem aconteceria, com decodificação restrita).

In [3]:
fake_classifier = keyword_llm(
    {
        "fatura": '{"intent": "billing", "confidence": 0.93, "rationale": "fala de fatura"}',
        "erro": '{"intent": "tech_support", "confidence": 0.88, "rationale": "relata erro"}',
        "cancelar": '{"intent": "cancellation", "confidence": 0.91, "rationale": "quer cancelar"}',
        "plano": '{"intent": "sales", "confidence": 0.55, "rationale": "talvez vendas"}',
        "boleto": '{"intent": "Cobrança", "confidence": 0.9, "rationale": "boleto"}',
    },
    default='{"intent": "other", "confidence": 0.4, "rationale": "sem pista"}',
)
messages = ["Minha fatura veio duplicada", "Dá erro 500 ao fazer login", "Quero cancelar tudo",
            "Qual plano serve para 3 pessoas?", "Bom dia!", "Não recebi o boleto"]

llm = ScriptedLLM([fake_classifier] * len(messages))
results = {}
for msg in messages:
    raw = llm([{"role": "user", "content": msg}], system=SYSTEM).text
    try:
        results[msg] = Classification.model_validate_json(raw)
        print(f"{msg:<36} -> {results[msg].intent!s:<13} ({results[msg].confidence:.2f})")
    except ValidationError as e:
        print(f"{msg:<36} -> INVÁLIDO: {e.errors(include_url=False)[0]['msg']}")

Minha fatura veio duplicada          -> billing       (0.93)
Dá erro 500 ao fazer login           -> tech_support  (0.88)
Quero cancelar tudo                  -> cancellation  (0.91)
Qual plano serve para 3 pessoas?     -> sales         (0.55)
Bom dia!                             -> other         (0.40)
Não recebi o boleto                  -> INVÁLIDO: Input should be 'billing', 'tech_support', 'sales', 'cancellation' or 'other'


## 2. Roteamento: tabela de despacho + limiar de confiança

`dict[Intent, handler]` substitui cadeias de `if`. Um teste garante que **todo** membro tem handler.
Abaixo do limiar de confiança, o caso vai para um humano em vez de ser roteado com base num chute.

In [4]:
def handle_billing(msg): return "→ agente de cobrança (com ferramentas de fatura/reembolso)"
def handle_tech(msg): return "→ agente de suporte técnico (busca na base de conhecimento)"
def handle_sales(msg): return "→ SDR humano + material de planos"
def handle_cancel(msg): return "→ fluxo de retenção"
def handle_other(msg): return "→ resposta genérica"

ROUTES: dict[Intent, Callable[[str], str]] = {
    Intent.BILLING: handle_billing,
    Intent.TECH_SUPPORT: handle_tech,
    Intent.SALES: handle_sales,
    Intent.CANCELLATION: handle_cancel,
    Intent.OTHER: handle_other,
}
assert set(ROUTES) == set(Intent), f"intenções sem rota: {set(Intent) - set(ROUTES)}"

CONFIDENCE_THRESHOLD = 0.7

def route(msg: str, c: Classification) -> str:
    if c.confidence < CONFIDENCE_THRESHOLD and c.intent is not Intent.OTHER:
        return f"→ fila humana (baixa confiança em {c.intent}: {c.confidence:.2f})"
    return ROUTES[c.intent](msg)

for msg, c in results.items():
    print(f"{msg:<36} {route(msg, c)}")

Minha fatura veio duplicada          → agente de cobrança (com ferramentas de fatura/reembolso)
Dá erro 500 ao fazer login           → agente de suporte técnico (busca na base de conhecimento)
Quero cancelar tudo                  → fluxo de retenção
Qual plano serve para 3 pessoas?     → fila humana (baixa confiança em sales: 0.55)
Bom dia!                             → resposta genérica


## 3. Multi-rótulo com `Flag`

Uma mensagem pode tocar vários assuntos ("cobraram errado **e** o app não abre"). O LLM devolve uma **lista** de rótulos
(fácil de restringir com schema); o código converte para `Flag` — e operações de conjunto ficam triviais.

In [5]:
class Topic(Flag):
    BILLING = auto()
    BUG = auto()
    PERFORMANCE = auto()
    PRIVACY = auto()
    FEATURE_REQUEST = auto()

TopicName = Literal[tuple(t.name.lower() for t in Topic)]     # schema: enum com os nomes

class TopicTags(BaseModel):
    topics: list[TopicName] = Field(min_length=1, max_length=3)

    def as_flag(self) -> Topic:
        return reduce(or_, (Topic[t.upper()] for t in self.topics), Topic(0))

TEAM_OWNERSHIP = {
    "finance": Topic.BILLING,
    "engineering": Topic.BUG | Topic.PERFORMANCE,
    "legal": Topic.PRIVACY,
    "product": Topic.FEATURE_REQUEST | Topic.PERFORMANCE,
}

tags = TopicTags.model_validate_json('{"topics": ["billing", "performance"]}').as_flag()
print(f"tópicos: {tags}")
print(f"times notificados: {[team for team, owned in TEAM_OWNERSHIP.items() if tags & owned]}")
print(f"precisa do jurídico?: {Topic.PRIVACY in tags}")
print(f'schema: {TopicTags.model_json_schema()["properties"]["topics"]["items"]}')

tópicos: Topic.BILLING|PERFORMANCE
times notificados: ['finance', 'engineering', 'product']
precisa do jurídico?: False
schema: {'enum': ['billing', 'bug', 'performance', 'privacy', 'feature_request'], 'type': 'string'}


## 4. Seleção de modelo por tier

Rotear tarefas simples para modelos menores é uma das maiores alavancas de custo e latência. O Enum guarda a tabela de decisão
(nome do modelo, preço, janela de contexto) junto do vocabulário.

> ⚠️ Nomes e preços abaixo são **ilustrativos** para o exercício — consulte a tabela de preços do seu provedor.

In [6]:
class ModelTier(Enum):
    #          (modelo,          US$/Mtok in, US$/Mtok out, contexto)
    FAST = ("small-model", 1.0, 5.0, 200_000)
    BALANCED = ("medium-model", 3.0, 15.0, 200_000)
    DEEP = ("large-model", 5.0, 25.0, 1_000_000)

    def __init__(self, model: str, usd_in: float, usd_out: float, context: int):
        self.model, self.usd_in, self.usd_out, self.context = model, usd_in, usd_out, context

    def cost(self, tokens_in: int, tokens_out: int) -> float:
        return tokens_in / 1e6 * self.usd_in + tokens_out / 1e6 * self.usd_out

    @classmethod
    def for_task(cls, intent: Intent, prompt_tokens: int) -> "ModelTier":
        if prompt_tokens > cls.BALANCED.context:
            return cls.DEEP
        return {Intent.OTHER: cls.FAST, Intent.SALES: cls.FAST,
                Intent.BILLING: cls.BALANCED, Intent.CANCELLATION: cls.BALANCED,
                Intent.TECH_SUPPORT: cls.DEEP}[intent]

workload = [(Intent.OTHER, 800, 150)] * 600 + [(Intent.BILLING, 3_000, 400)] * 300 + [(Intent.TECH_SUPPORT, 12_000, 900)] * 100
routed = sum(ModelTier.for_task(i, tin).cost(tin, tout) for i, tin, tout in workload)
all_deep = sum(ModelTier.DEEP.cost(tin, tout) for _, tin, tout in workload)
print(f"tudo no DEEP: US$ {all_deep:.2f} | roteado por tier: US$ {routed:.2f} | economia: {1 - routed / all_deep:.0%}")

tudo no DEEP: US$ 20.40 | roteado por tier: US$ 13.68 | economia: 33%


## 5. Guardrails: capacidades do agente com `Flag`

Cada ferramenta declara as **capacidades** que exige; cada perfil de agente declara as que **possui**. Antes de executar uma
chamada pedida pelo LLM, o código verifica `required in granted` — o modelo não consegue "convencer" o código a liberar.

In [7]:
class Capability(Flag):
    READ_DATA = auto()
    WRITE_DATA = auto()
    SEND_EMAIL = auto()
    NETWORK = auto()
    MONEY = auto()
    EXECUTE_CODE = auto()

    READ_ONLY = READ_DATA | NETWORK
    SUPPORT = READ_DATA | WRITE_DATA | SEND_EMAIL

TOOL_REQUIREMENTS: dict[str, Capability] = {
    "search_kb": Capability.READ_DATA,
    "web_fetch": Capability.NETWORK,
    "update_ticket": Capability.WRITE_DATA,
    "email_customer": Capability.SEND_EMAIL | Capability.READ_DATA,
    "issue_refund": Capability.MONEY | Capability.WRITE_DATA,
    "run_python": Capability.EXECUTE_CODE,
}

AGENT_PROFILES = {
    "researcher": Capability.READ_ONLY,
    "support_agent": Capability.SUPPORT,
    "billing_agent": Capability.SUPPORT | Capability.MONEY,
}

class PermissionDenied(Exception):
    pass

def authorize(agent: str, tool: str) -> None:
    granted, required = AGENT_PROFILES[agent], TOOL_REQUIREMENTS[tool]
    if required not in granted:
        missing = required & ~granted
        raise PermissionDenied(f"{agent} não pode usar {tool}: faltam {missing}")

for agent, tool in [("researcher", "web_fetch"), ("researcher", "update_ticket"),
                    ("support_agent", "email_customer"), ("support_agent", "issue_refund"),
                    ("billing_agent", "issue_refund"), ("billing_agent", "run_python")]:
    try:
        authorize(agent, tool)
        print(f"✔ {agent:<14} {tool}")
    except PermissionDenied as e:
        print(f"✘ {e}")

# só expomos ao LLM as ferramentas que o perfil pode usar (menos tentação, menos tokens)
visible = {a: [t for t, req in TOOL_REQUIREMENTS.items() if req in g] for a, g in AGENT_PROFILES.items()}
print(f"json.dumps(visible, indent=1):\n{json.dumps(visible, indent=1)}")

✔ researcher     web_fetch
✘ researcher não pode usar update_ticket: faltam Capability.WRITE_DATA
✔ support_agent  email_customer
✘ support_agent não pode usar issue_refund: faltam Capability.MONEY
✔ billing_agent  issue_refund
✘ billing_agent não pode usar run_python: faltam Capability.EXECUTE_CODE
json.dumps(visible, indent=1):
{
 "researcher": [
  "search_kb",
  "web_fetch"
 ],
 "support_agent": [
  "search_kb",
  "update_ticket",
  "email_customer"
 ],
 "billing_agent": [
  "search_kb",
  "update_ticket",
  "email_customer",
  "issue_refund"
 ]
}


## 6. Enums "abertos" para respostas do provedor

O `stop_reason` diz **por que** o modelo parou — e o código precisa reagir diferente a cada um (resposta completa, cortada por limite
de tokens, pedido de ferramenta, recusa...). Provedores adicionam valores novos ao longo do tempo: um Enum fechado quebraria
o agente em produção no dia em que um valor novo aparecer. `_missing_` → `UNKNOWN` + log mantém o sistema de pé.

In [8]:
import logging
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s", force=True)

class StopReason(StrEnum):
    END_TURN = "end_turn"
    MAX_TOKENS = "max_tokens"
    STOP_SEQUENCE = "stop_sequence"
    TOOL_USE = "tool_use"
    PAUSE_TURN = "pause_turn"
    REFUSAL = "refusal"
    UNKNOWN = "unknown"

    @classmethod
    def _missing_(cls, value):
        logging.warning("stop_reason desconhecido: %r", value)
        return cls.UNKNOWN

def next_step(raw_stop_reason: str) -> str:
    match StopReason(raw_stop_reason):
        case StopReason.END_TURN | StopReason.STOP_SEQUENCE:
            return "entregar resposta"
        case StopReason.TOOL_USE:
            return "executar ferramentas e continuar o loop"
        case StopReason.MAX_TOKENS:
            return "resposta truncada: pedir continuação ou aumentar max_tokens (NÃO parsear como JSON completo)"
        case StopReason.PAUSE_TURN:
            return "reenviar a conversa para o modelo continuar"
        case StopReason.REFUSAL:
            return "não usar o conteúdo; resposta padrão + log"
        case StopReason.UNKNOWN:
            return "tratar como falha recuperável + alerta"

for r in ["end_turn", "tool_use", "max_tokens", "refusal", "brand_new_reason"]:
    print(f"{r:<17} -> {next_step(r)}")

WARNING stop_reason desconhecido: 'brand_new_reason'


end_turn          -> entregar resposta
tool_use          -> executar ferramentas e continuar o loop
max_tokens        -> resposta truncada: pedir continuação ou aumentar max_tokens (NÃO parsear como JSON completo)
refusal           -> não usar o conteúdo; resposta padrão + log
brand_new_reason  -> tratar como falha recuperável + alerta


## 7. Checklist

| ✔ | prática |
|---|---|
| ☐ | Todo campo categórico da saída do LLM é um Enum/`Literal` (com opção `other`/`unknown`) |
| ☐ | Descrições dos rótulos moram no Enum e geram o prompt — uma fonte da verdade |
| ☐ | `dict[Enum, handler]` + teste de cobertura (`set(ROUTES) == set(Intent)`) |
| ☐ | Limiar de confiança com fallback humano para decisões de baixa confiança |
| ☐ | Multi-rótulo: lista no schema, `Flag` no código |
| ☐ | Tiers de modelo como Enum com metadados de custo; roteamento por tarefa |
| ☐ | Guardrails por `Flag` de capacidades, checados no código antes de executar qualquer ferramenta |
| ☐ | Valores vindos do provedor (`stop_reason`, códigos de erro) como Enum **aberto** com `UNKNOWN` |

➡️ Próximo: [`04_dataclass_agent_runtime.ipynb`](04_dataclass_agent_runtime.ipynb)